In [14]:
import pandas as pd
import numpy as np
from utils.feature_group import load_feature_groups

In [15]:
df = pd.read_csv('../data/clinical/december2025.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2247 entries, 0 to 2246
Columns: 2517 entries, PTID to AB_SUVR_WMGM_CENT_CT_TRNSFRMD
dtypes: float64(2386), int64(69), object(62)
memory usage: 43.1+ MB


/tmp/ipykernel_414631/1542233327.py:1: DtypeWarning: Columns (0,297,305,512,915,917,919,921,1092,1097,1102,1106,1314,1432,1455,1741,1743,1824,1825,1826,1830,1871) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/clinical/december2025.csv')


# 1.Pre-processing

In [16]:
df = df[~df['PTID'].str.startswith('A', na=False)]

## MMSE Processing
There are two MMSE columns and I would noticed that some of 'FL_MMSE' had a value of 999,-4, or empty so i just use the column 'MMSE' as a fall back to fill them.

Filling out MMSE Values. Using FL_MMSE Column as is the most complete, if the value is 999 or -4 set it to NaN. 
fallback to MMSE values if is missing.

In [17]:
df["COMBINED_MMSE"] = df["FL_MMSE"].replace([999, -4], np.nan)
df["COMBINED_MMSE"] = df["COMBINED_MMSE"].fillna(df["NACCMMSE"])
df = df.drop(columns=["FL_MMSE", "NACCMMSE"])

In [18]:
df = df.rename(columns={"COMBINED_NE4S": "APOE4S", "COMBINED_MMSE": "MMSE"})

In [19]:
FEATURE_GROUPS = load_feature_groups()

In [20]:
FEATURES = [col for group in FEATURE_GROUPS.values() for col in group]
FEATURES[:10]

['PTID',
 'VISITYR',
 'NACCAGE',
 'SEX',
 'EDUC',
 'CDRSUM',
 'MMSE',
 'HVLT_DR',
 'LASSI_A_CR2',
 'LASSI_A_CR2_INT']

In [21]:
df_filter = df[FEATURES + ["VOL_ETIV"]].copy()

In [22]:
vol_cols = [c for c in df_filter.columns if c.startswith("VOL_") and c != "VOL_ETIV"]

In [24]:
df_filter[vol_cols] = df_filter[vol_cols].div(df_filter["VOL_ETIV"], axis=0)

In [25]:
df_filter = df_filter.dropna(subset=df_filter.columns.difference(["NACCETPR"]))

In [26]:
df_filter.info()

<class 'pandas.core.frame.DataFrame'>
Index: 457 entries, 6 to 2223
Columns: 278 entries, PTID to VOL_ETIV
dtypes: float64(274), int64(3), object(1)
memory usage: 996.1+ KB


In [27]:
df_filter.drop(columns=["VOL_ETIV"], inplace=True)

In [28]:
# WENT FROM 591 ROWS TO 394

In [29]:
# drop_df_filter = df_filter.dropna(subset=["CDRGLOB", "AMYLPET","FL_UDSD", "NACCETPR"])

In [30]:
# drop_df_filter.info()

In [31]:
df_filter["NACCETPR"].value_counts(dropna=False)

NACCETPR
1.0     143
NaN      91
18.0     64
30.0     27
19.0     27
2.0      25
8.0      21
22.0     12
21.0     10
25.0      6
26.0      6
13.0      5
20.0      5
7.0       4
27.0      3
29.0      3
9.0       2
24.0      1
17.0      1
28.0      1
Name: count, dtype: int64

In [32]:
df_filter.to_csv('../data/clinical/preprocess.csv', index=False)